# Emergency Department Arrival Forecasting

## Project Overview

Forecasts **daily ED arrivals** over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, flu-season surges, Monday spikes, and holiday effects.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily ED arrival counts, predict the next 14 days. ED arrivals drive real-time staffing, bed flow, and ambulance diversion decisions.

## Why This Project Matters

EDs operate at the edge of capacity. Accurate arrival forecasts prevent overcrowding, reduce wait times, and improve patient safety. Under-forecasting leads to boarding, diversion, and adverse outcomes.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily ED arrivals
- Weekly cycle (Monday peak, weekend variation)
- Flu season increase (Dec-Mar)
- Holiday dips (Christmas, Thanksgiving)
- Gradual upward trend

| Column | Description |
|--------|-------------|
| `date` | Date |
| `arrivals` | Daily ED arrival count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'arrivals'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 300 + 0.1 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow == 0: weekly[i] = 40  # Monday peak
    elif dow <= 4: weekly[i] = 20
    else: weekly[i] = -20

flu = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m = dates[i].month
    if m in [12, 1, 2, 3]: flu[i] = 40

holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and d >= 24) or (m == 1 and d <= 2): holiday[i] = -50
    elif m == 11 and 22 <= d <= 28: holiday[i] = -30

noise = np.random.normal(0, 20, N_POINTS)
arrivals = base + weekly + flu + holiday + noise
arrivals = np.maximum(arrivals, 100).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'arrivals': arrivals})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,arrivals
0,2022-01-01,280
1,2022-01-02,267
2,2022-01-03,393
3,2022-01-04,391
4,2022-01-05,356


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('arrivals Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("arrivals Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

arrivals Statistics:
count    730.000000
mean     358.828767
std       37.191227
min      267.000000
25%      334.000000
50%      358.000000
75%      385.000000
max      485.000000
Name: arrivals, dtype: float64

CV: 0.104
Skewness: 0.044


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       39.4 | RMSE:       47.8 | MAPE: 10.92%
Seasonal Naive (7)             | MAE:       31.8 | RMSE:       40.8 | MAPE:  8.66%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared   R-Squared        RMSE  Time Taken
Model                                                                           
GaussianProcessRegressor         2304.601706 -176.200131  399.626849    0.086556
KernelRidge                      2071.273396 -158.251800  378.847838    0.063111
MLPRegressor                      134.194631   -9.245741   96.093569    0.539661
QuantileRegressor                  72.162829   -4.474064   70.238878    0.068511
DummyRegressor                     69.548322   -4.272948   68.936524    0.021836
ExtraTreeRegressor                 56.453616   -3.265663   62.003456    0.032638
NuSVR                              55.942773   -3.226367   61.717205    0.031748
DecisionTreeRegressor              53.621280   -3.047791   60.399267    0.047681
SVR                                52.877855   -2.990604   59.971094    0.040860
XGBRegressor                       40.008418   -2.000648   52.003209    5.849763


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:       40.9 | RMSE:       46.9 | MAPE:  9.48%
FLAML Test (catboost)          | MAE:       29.7 | RMSE:       34.8 | MAPE:  7.33%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       35.5 | RMSE:       45.1 | MAPE: 10.03%
SF AutoETS                     | MAE:       33.9 | RMSE:       41.9 | MAPE:  9.48%
SF SeasonalNaive               | MAE:       41.3 | RMSE:       50.9 | MAPE: 11.43%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
FLAML Test (catboost) 29.657592 34.781550  7.331375
   Seasonal Naive (7) 31.785714 40.798284  8.660151
           SF AutoETS 33.943752 41.885739  9.479130
         SF AutoARIMA 35.522057 45.126815 10.026558
   Naive (Last Value) 39.428571 47.795696 10.921823
     FLAML (catboost) 40.912752 46.857702  9.484072
     SF SeasonalNaive 41.285713 50.943946 11.427962

Top 3:
                model       MAE      RMSE     MAPE
FLAML Test (catboost) 29.657592 34.781550 7.331375
   Seasonal Naive (7) 31.785714 40.798284 8.660151
           SF AutoETS 33.943752 41.885739 9.479130


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 20.15, Std: 28.35


## Interpretation and Business Insight

- Monday arrivals are highest (deferred weekend complaints)
- Flu season adds ~15% to baseline volume
- Holiday dips are real but short-lived
- Growth trend reflects population and ED-as-safety-net dynamics
- Lag features at 7-day intervals are most predictive

## Limitations

1. Synthetic — real EDs have acuity mix, walk-in vs ambulance
2. No hourly granularity (intraday patterns matter)
3. No triage category segmentation
4. Weather effects not modeled

## How to Improve This Project

1. Add hourly forecasting for shift-level staffing
2. Segment by triage category (ESI 1-5)
3. Add weather and local event data
4. Model boarding time and throughput
5. Use real-time updates as morning data arrives

## Production Considerations

- Hourly retraining with intraday updates
- Integration with ED tracking boards
- Real-time diversion trigger based on forecast vs capacity
- Dashboard showing predicted wait times

## Common Mistakes

1. Using daily forecasts for intraday staffing decisions
2. Ignoring flu season in planning
3. Not planning for post-holiday catch-up surges
4. Treating all arrivals equally (acuity matters)

## Mini Challenge / Exercises

1. Add weather data (temperature, rain) and measure impact
2. Try hourly forecasting for a subset of the data
3. Build a queuing model on top of arrival forecasts
4. Simulate diversion decisions based on forecasted volume

## Final Summary / Key Takeaways

1. ED arrival forecasting is critical for patient safety
2. Weekly patterns (Monday peak) and flu season drive most variation
3. 14-day forecasts enable staffing and resource planning
4. Holiday effects require special handling
5. Real-world deployment needs hourly granularity

In [18]:
import json
metrics = {
    'project': 'Emergency Department Arrival Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Emergency Department Arrival Forecasting — Complete ===")

Metrics saved.

=== Emergency Department Arrival Forecasting — Complete ===
